In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

In [2]:
df=pd.read_csv("heart_failure_clinical_records_dataset.csv")

In [3]:
df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [4]:
df["DEATH_EVENT"].value_counts()

,count
DEATH_EVENT,
0,203
1,96


In [5]:
#Data Inspection
print(df.shape)
print(df.columns)
print(df.describe())
print(df.info())

(299, 13)
Index(['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes',
       'ejection_fraction', 'high_blood_pressure', 'platelets',
       'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time',
       'DEATH_EVENT'],
      dtype='object')
              age     anaemia  creatinine_phosphokinase    diabetes  \
count  299.000000  299.000000                299.000000  299.000000   
mean    60.833893    0.431438                581.839465    0.418060   
std     11.894809    0.496107                970.287881    0.494067   
min     40.000000    0.000000                 23.000000    0.000000   
25%     51.000000    0.000000                116.500000    0.000000   
50%     60.000000    0.000000                250.000000    0.000000   
75%     70.000000    1.000000                582.000000    1.000000   
max     95.000000    1.000000               7861.000000    1.000000   

       ejection_fraction  high_blood_pressure      platelets  \
count         299.000000           299.0000

In [6]:
#checking null values
df.isnull().sum()

,0
age,0
anaemia,0
creatinine_phosphokinase,0
diabetes,0
ejection_fraction,0
high_blood_pressure,0
platelets,0
serum_creatinine,0
serum_sodium,0
sex,0


In [7]:
X=df.drop(["DEATH_EVENT"],axis=1)
y=df["DEATH_EVENT"]

In [8]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [9]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
scaler.fit(X_train)
X_train_scaled=scaler.transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [10]:
models={
    "Logistic Regression":LogisticRegression(),
    "Navie Bayes":GaussianNB(),
    "KNN":KNeighborsClassifier(n_neighbors=3),
    "Decision Tree":DecisionTreeClassifier(),
    "SVM":SVC(probability=True)
}


In [11]:
result=[]

In [12]:

for name, model in models.items():
  model.fit(X_train_scaled,y_train)
  y_pred=model.predict(X_test_scaled)
  accuracy=accuracy_score(y_test,y_pred)
  f1score=f1_score(y_test,y_pred)
  result.append({
      "model":name,"Accuracy":round(accuracy,4),"F1-Score":round(f1score,4)
  })


In [13]:
result

[{'model': 'Logistic Regression', 'Accuracy': 0.8, 'F1-Score': 0.7},
 {'model': 'Navie Bayes', 'Accuracy': 0.7, 'F1-Score': 0.5},
 {'model': 'KNN', 'Accuracy': 0.7333, 'F1-Score': 0.5556},
 {'model': 'Decision Tree', 'Accuracy': 0.7167, 'F1-Score': 0.6222},
 {'model': 'SVM', 'Accuracy': 0.75, 'F1-Score': 0.6154}]

In [14]:
import joblib
joblib.dump(models["Logistic Regression"],"Logistic_heart_pred.pkl")


['Logistic_heart_pred.pkl']

In [15]:
joblib.dump(scaler,"scaler.pkl")
joblib.dump(X.columns.tolist(),"columns.pkl")

['columns.pkl']

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
import joblib

# -------------------------
# Split with stratification
# -------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------
# Scale data
# -------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------
# Train Proper Logistic Model
# -------------------------
model = LogisticRegression(
    class_weight='balanced',   # 🔥 KEY FIX
    max_iter=1000
)

model.fit(X_train_scaled, y_train)

# -------------------------
# Evaluate Correctly
# -------------------------
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:,1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

# -------------------------
# Save model artifacts
# -------------------------
joblib.dump(model, "Logistic_heart_pred.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(X.columns.tolist(), "columns.pkl")


              precision    recall  f1-score   support

           0       0.82      0.90      0.86        41
           1       0.73      0.58      0.65        19

    accuracy                           0.80        60
   macro avg       0.78      0.74      0.75        60
weighted avg       0.79      0.80      0.79        60

ROC-AUC: 0.8549422336328627


['columns.pkl']